# 05 — Fine-tune on Diverse Real-World Faces

Adds UTKFace (diverse demographics, outdoor/indoor real photos) to the real class so the model
stops flagging dark-skinned and low-light real photos as fake.

**Before running:**
1. Right panel → Accelerator → GPU T4 x2
2. Right panel → Add Data → search `celebdf-v2image-dataset` → Add
3. Right panel → Add Data → search `utkface-new` → Add (by jangedoo)
4. Right panel → Add Data → search `deepsentinel-weights` → Add (or we download via HTTP)

In [ ]:
import torch
from pathlib import Path

assert torch.cuda.is_available(), 'GPU not enabled! Right panel → Accelerator → GPU T4 x2'
print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda'

# Locate Celeb-DF V2
SEARCH_ROOTS = [
    Path('/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2'),
    Path('/kaggle/input/celebdf-v2image-dataset/Celeb_V2'),
]
DATA_ROOT = next((p for p in SEARCH_ROOTS if (p / 'Train').exists()), None)
assert DATA_ROOT, 'Celeb-DF V2 not found — add it via right panel → Add Data'
print(f'Celeb-DF root: {DATA_ROOT}')

# Locate UTKFace
UTK_SEARCH = [
    Path('/kaggle/input/utkface-new/UTKFace'),
    Path('/kaggle/input/utkface-new/utkface_aligned_cropped/UTKFace'),
    Path('/kaggle/input/utkface-new'),
]
UTK_DIR = next((p for p in UTK_SEARCH if p.exists() and any(p.iterdir())), None)
assert UTK_DIR, 'UTKFace not found — add it via right panel → Add Data → search utkface-new'
utk_images = list(UTK_DIR.glob('*.jpg')) + list(UTK_DIR.glob('*.png'))
print(f'UTKFace: {len(utk_images):,} images at {UTK_DIR}')

In [ ]:
# Download existing model weights from HuggingFace
import requests

WEIGHTS_URL  = 'https://huggingface.co/Oveea/deepsentinel-weights/resolve/main/efficientnet_b4.pth'
WEIGHTS_BASE = Path('/kaggle/working/efficientnet_b4_base.pth')
WEIGHTS_OUT  = Path('/kaggle/working/efficientnet_b4.pth')

if not WEIGHTS_BASE.exists():
    print('Downloading base weights...')
    with requests.get(WEIGHTS_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(WEIGHTS_BASE, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    print(f'Done: {WEIGHTS_BASE.stat().st_size / 1e6:.1f} MB')
else:
    print('Weights already downloaded.')

In [ ]:
# Build mixed dataset
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from PIL import Image

IMAGE_SIZE = 224
BATCH_SIZE = 32
UTK_SAMPLE  = 5000   # how many UTKFace real images to add

train_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

infer_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class UTKFaceDataset(Dataset):
    """Wraps UTKFace images as class 0 (real), matching Celeb-real=0."""
    def __init__(self, image_paths, transform=None):
        self.paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, 0  # label 0 = real (same as Celeb-real)


# Celeb-DF training set (real=0, synthesis=1)
celeb_train = ImageFolder(str(DATA_ROOT / 'Train'), transform=train_transforms)
celeb_val   = ImageFolder(str(DATA_ROOT / 'Val'),   transform=infer_transforms)
print(f'Celeb-DF classes: {celeb_train.classes}  (real=0 expected)')
print(f'Celeb-DF train: {len(celeb_train):,}  val: {len(celeb_val):,}')

# Sample UTKFace real images
random.seed(42)
utk_sample = random.sample(utk_images, min(UTK_SAMPLE, len(utk_images)))
utk_train  = UTKFaceDataset(utk_sample, transform=train_transforms)
print(f'UTKFace sample added to real class: {len(utk_train):,}')

# Combined training set
train_ds = ConcatDataset([celeb_train, utk_train])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(celeb_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f'Total training samples: {len(train_ds):,}')

In [ ]:
# Load model with existing weights
import sys
sys.path.append('/kaggle/working')  # adjust if src/ is elsewhere

import timm
import torch.nn as nn

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(self.backbone.num_features, 1),
        )
    def forward(self, x):
        return self.classifier(self.backbone(x))

model = DeepfakeDetector().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_BASE, map_location=DEVICE))
print('Loaded base weights.')

In [ ]:
# Fine-tune — low learning rate so existing knowledge is preserved
import torch.optim as optim

EPOCHS   = 5
LR       = 2e-5   # 5x lower than original training — preserves existing weights

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    # --- validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            val_loss += criterion(model(imgs), labels).item() * imgs.size(0)
    val_loss /= len(celeb_val)
    scheduler.step()

    print(f'Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), WEIGHTS_OUT)
        print(f'  ✓ Saved best model (val_loss={val_loss:.4f})')

print(f'\nFine-tuning complete. Best val_loss: {best_val_loss:.4f}')
print(f'Weights saved to: {WEIGHTS_OUT}')

In [ ]:
# Quick sanity check — evaluate on validation set
from sklearn.metrics import classification_report, roc_auc_score

model.load_state_dict(torch.load(WEIGHTS_OUT, map_location=DEVICE))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).squeeze(1).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

preds = (np.array(all_probs) >= 0.5).astype(int)
auc   = roc_auc_score(all_labels, all_probs)
print(f'Val AUC: {auc:.4f}')
print(classification_report(all_labels, preds, target_names=celeb_val.classes))

In [ ]:
# Upload new weights to HuggingFace
# Replace HF_TOKEN with your HuggingFace token (Settings → Access Tokens)
import subprocess, os

HF_TOKEN   = ''   # paste your token here
HF_REPO_ID = 'Oveea/deepsentinel-weights'

assert HF_TOKEN, 'Paste your HuggingFace token above!'

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=str(WEIGHTS_OUT),
    path_in_repo='efficientnet_b4.pth',
    repo_id=HF_REPO_ID,
    repo_type='model',
)
print(f'Uploaded to https://huggingface.co/{HF_REPO_ID}')
print('Now do a Factory Rebuild on your HuggingFace Space to pick up the new weights.')